<a href="https://colab.research.google.com/github/3160664535qqcom-ctrl/kangtai-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** 15:34 **Stopped at:** 16:34  **Furthest step I reached (see the brief):** gold_table

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [2]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [3]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Add text here

In [4]:
# 1. Payments: Check for duplicate payment_id records
payments_audit = q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT payment_id) AS unique_payments,
        COUNT(*) - COUNT(DISTINCT payment_id) AS repeated_payments
    FROM payments
""")
print("--- Payments Uniqueness Audit ---")
print(payments_audit.to_string(index=False))

# 2. Orders: Check order_id uniqueness and delivery_fee statistics
orders_audit = q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders,
        MIN(delivery_fee) AS min_fee,
        AVG(delivery_fee) AS avg_fee,
        MAX(delivery_fee) AS max_fee,
        APPROX_QUANTILE(delivery_fee, 0.99) AS p99_fee
    FROM orders
""")
print("\n--- Orders & Delivery Fee Audit ---")
print(orders_audit.to_string(index=False))

# 3. Order Lines: List some unique item names to spot check
order_lines_audit = q("""
    SELECT DISTINCT item_name
    FROM order_lines
    ORDER BY item_name
    LIMIT 15
""")
print("\n--- Sample Item Names (Order Lines) ---")
print(order_lines_audit.to_string(index=False))

# 4. Customers: Check home_city spellings and customer_id uniqueness
customers_audit = q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT customer_id) AS unique_customers
    FROM customers
""")
print("\n--- Customers Uniqueness Audit ---")
print(customers_audit.to_string(index=False))

city_spellings = q("SELECT DISTINCT home_city FROM customers ORDER BY home_city")
print("\n--- Customer Home Cities Spellings ---")
print(city_spellings.to_string(index=False))

# 5. Restaurants: Check for repeated restaurant_id records
restaurants_audit = q("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT restaurant_id) AS unique_restaurants
    FROM restaurants
""")
print("\n--- Restaurants Uniqueness Audit ---")
print(restaurants_audit.to_string(index=False))

--- Payments Uniqueness Audit ---
 total_rows  unique_payments  repeated_payments
      16423            16393                 30

--- Orders & Delivery Fee Audit ---
 total_rows  unique_orders  min_fee  avg_fee  max_fee  p99_fee
      14494          14494      0.0 2.150676     3.49     3.49

--- Sample Item Names (Order Lines) ---
    item_name
      Burrito
 Cheeseburger
Chicken tikka
      Dessert
 Falafel wrap
   Fried rice
        Fries
   Margherita
     Pad thai
        Ramen
   Side salad
   Soft drink

--- Customers Uniqueness Audit ---
 total_rows  unique_customers
       4200              4200

--- Customer Home Cities Spellings ---
 home_city
Birmingham
   Bristol
     Leeds
    London
Manchester

--- Restaurants Uniqueness Audit ---
 total_rows  unique_restaurants
         67                  64


In [6]:
# 1. Examine repeated restaurant_id records (SCD Type 2 check)
repeated_restaurants = q("""
    SELECT restaurant_id, restaurant_name, city, cuisine, effective_from, COUNT(*) OVER(PARTITION BY restaurant_id) as cnt
    FROM restaurants
    WHERE restaurant_id IN (
        SELECT restaurant_id FROM restaurants GROUP BY restaurant_id HAVING COUNT(*) > 1
    )
    ORDER BY restaurant_id, effective_from
""")
print("--- Repeated Restaurant IDs (Slowly Changing Dimensions) ---")
print(repeated_restaurants.to_string(index=False))

# 2. Check unique spellings of cities and cuisines in restaurants
restaurants_spelling = q("""
    SELECT DISTINCT city, cuisine
    FROM restaurants
    ORDER BY city, cuisine
""")
print("\n--- Restaurants Cities & Cuisines Spellings ---")
print(restaurants_spelling.to_string(index=False))

# Helper check: Ensure clean tables are registered before querying them
existing_tables = con.sql("SHOW TABLES").df()['name'].tolist()

if 'clean_order_status_events' in existing_tables:
    # 3. Get the latest status of each order and count different final states
    status_distribution = q("""
        SELECT final_status, COUNT(*) AS count
        FROM clean_order_status_events
        GROUP BY final_status
    """)
    print("\n--- Distribution of Final Order Statuses ---")
    print(status_distribution.to_string(index=False))
else:
    print("\n[提示] clean_order_status_events 表尚未在 DuckDB 中注册。请先运行上方的清洗单元格 (cell 006e0d76)。")

if 'clean_orders' in existing_tables and 'clean_order_status_events' in existing_tables and 'clean_payments' in existing_tables:
    # 4. Link payments to order status: see if unmatched orders exist
    payment_status_match = q("""
        SELECT
            ev.final_status,
            COUNT(o.order_id) AS total_orders,
            COUNT(p.order_id) AS orders_with_payments,
            COUNT(o.order_id) - COUNT(p.order_id) AS orders_without_payments
        FROM clean_orders o
        LEFT JOIN clean_order_status_events ev ON o.order_id = ev.order_id
        LEFT JOIN clean_payments p ON o.order_id = p.order_id
        GROUP BY ev.final_status
    """)
    print("\n--- Payment and Order Status Reconciliation ---")
    print(payment_status_match.to_string(index=False))
else:
    print("[提示] clean_orders 或 clean_payments 表尚未在 DuckDB 中注册。请先运行完整的 Silver 清洗步骤单元格。")

--- Repeated Restaurant IDs (Slowly Changing Dimensions) ---
restaurant_id  restaurant_name       city  cuisine effective_from  cnt
         R007 Royal Japanese 7    Bristol Japanese     2025-01-01    2
         R007 Royal Japanese 7     London Japanese     2026-04-14    2
         R021    Grand Thai 21 Manchester     Thai     2025-01-01    2
         R021    Grand Thai 21      LEEDS     Thai     2026-05-06    2
         R040  Corner Vegan 40     London    Vegan     2025-01-01    2
         R040  Corner Vegan 40    Bristol    Vegan     2026-05-27    2

--- Restaurants Cities & Cuisines Spellings ---
      city   cuisine
Birmingham   Burgers
Birmingham   Chinese
Birmingham    Indian
Birmingham   Italian
Birmingham  Italian 
Birmingham  Japanese
Birmingham Japanese 
Birmingham      Thai
Birmingham     Vegan
   Bristol   Burgers
   Bristol   Chinese
   Bristol  Japanese
   Bristol     Vegan
     LEEDS      Thai
    LONDON    Indian
     Leeds   Burgers
     Leeds   Italian
     Leeds   Me

In [ ]:
#  order liens dosen't have unique id, but includes items
# status table has 4 stage of status
# orders table showed the delivery fee
# payment table has refunded item
# restaurant table has cuisine

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

Add text here

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

In [ ]:
#  restaurunt table: standardrize them (formats of the 5 cities should be the same, cuisine, restaurant_name)

#  orders table :to see if order_id are unique,

# Order lines: collapse to one number per order (quantity times price,
# summed) so it can’t multiply rows when joined.

# order_status_events : remove exact duplicates, then keep the earliest event per order and status.
# Pivot to one row per order with a column for each timestamp (placed, accepted, picked up, delivered, cancelled).
#  Add a clear final status: delivered, cancelled, or no outcome.

# customers: formats of the 5 cities should be the same

#Payments: remove the 30 duplicate rows, then split charges from refunds. Sum each to one charged amount and one refunded amount per order.

#Quality flags: don’t delete suspicious rows.
# Mark them (for example “negative duration” or “over 3 hours”)
# so you can count them and explain them.

In [4]:
import numpy as np
import pandas as pd

# 1. Clean Restaurants
restaurants_df = tables['restaurants'].copy()
restaurants_df['city'] = restaurants_df['city'].str.strip().str.title()
restaurants_df['cuisine'] = restaurants_df['cuisine'].str.strip().str.title()
restaurants_df['restaurant_name'] = restaurants_df['restaurant_name'].str.strip()
con.register('clean_restaurants', restaurants_df)

# 2. Clean Orders
orders_df = tables['orders'].copy()
orders_df['order_ts'] = pd.to_datetime(orders_df['order_ts'])
is_orders_unique = orders_df['order_id'].is_unique
print(f"Are order_ids unique in orders table? {is_orders_unique}")
con.register('clean_orders', orders_df)

# 3. Clean Order Lines
order_lines_df = tables['order_lines'].copy()
order_lines_df['line_total'] = order_lines_df['quantity'] * order_lines_df['unit_price']
order_lines_summary = order_lines_df.groupby('order_id').agg(
    total_items_quantity=('quantity', 'sum'),
    total_items_cost=('line_total', 'sum')
).reset_index()
con.register('clean_order_lines', order_lines_summary)

Are order_ids unique in orders table? True


In [7]:
# 4. Clean Order Status Events
events_df = tables['order_status_events'].copy()
events_df['event_ts'] = pd.to_datetime(events_df['event_ts'])

# Remove exact duplicates
events_df = events_df.drop_duplicates()

# Keep earliest event per order and status
events_df = events_df.sort_values(by=['order_id', 'status', 'event_ts'])
events_df = events_df.drop_duplicates(subset=['order_id', 'status'], keep='first')

# Pivot to one row per order
events_pivot = events_df.pivot(index='order_id', columns='status', values='event_ts').reset_index()

# Ensure all possible status columns exist
status_cols = ['placed', 'accepted', 'picked up', 'delivered', 'cancelled']
for col in status_cols:
    if col not in events_pivot.columns:
        events_pivot[col] = pd.NaT

# Add clear final status
def determine_final_status(row):
    if pd.notnull(row['cancelled']):
        return 'cancelled'
    elif pd.notnull(row['delivered']):
        return 'delivered'
    else:
        return 'no outcome'

events_pivot['final_status'] = events_pivot.apply(determine_final_status, axis=1)

# Quality Flags
# Calculate delivery duration in minutes
events_pivot['duration_minutes'] = (events_pivot['delivered'] - events_pivot['placed']).dt.total_seconds() / 60.0

events_pivot['flag_negative_duration'] = events_pivot['duration_minutes'] < 0
events_pivot['flag_over_3_hours'] = events_pivot['duration_minutes'] > 180

con.register('clean_order_status_events', events_pivot)

In [8]:
# 5. Clean Customers
customers_df = tables['customers'].copy()
customers_df['home_city'] = customers_df['home_city'].str.strip().str.title()
con.register('clean_customers', customers_df)

# 6. Clean Payments
payments_df = tables['payments'].copy()

# Remove duplicate rows
payments_df = payments_df.drop_duplicates()

# Split charges and refunds
payments_df['charged_amount'] = np.where(payments_df['payment_type'] == 'charge', payments_df['amount'], 0.0)
payments_df['refunded_amount'] = np.where(payments_df['payment_type'] == 'refund', payments_df['amount'], 0.0)

# Sum each per order_id
payments_summary = payments_df.groupby('order_id').agg(
    total_charged=('charged_amount', 'sum'),
    total_refunded=('refunded_amount', 'sum')
).reset_index()

con.register('clean_payments', payments_summary)

print("Data cleaning steps defined and registered with DuckDB!")

Data cleaning steps defined and registered with DuckDB!


In [ ]:
# Aggregate every child table (lines, events, payments) to one row per order first.
# This is the main protection against double-counting.

In [ ]:
# Join those onto the orders table.
# Join the restaurant by looking up the version that was valid at the time of the order, not just by ID.
# Take city from the restaurant, not from the customer’s home_city. Operations wants performance by the city where the food is made and delivered.

In [12]:
# We join the aggregated tables (clean_order_lines, clean_order_status_events, clean_payments)
# onto clean_orders along with clean_restaurants (using a rolling or matching historical date constraint)
join_query = """
SELECT
    o.order_id,
    o.customer_id,
    o.restaurant_id,
    o.order_ts,
    o.delivery_fee,

    -- Restaurant details (joined by matching historical validity)
    r.restaurant_name,
    r.cuisine,
    r.city AS restaurant_city,

    -- Aggregated order lines
    ol.total_items_quantity,
    ol.total_items_cost,

    -- Order status events
    ev.placed,
    ev.accepted,
    ev."picked up" AS picked_up,
    ev.delivered,
    ev.cancelled,
    ev.final_status,
    ev.duration_minutes,
    ev.flag_negative_duration,
    ev.flag_over_3_hours,

    -- Aggregated payments
    p.total_charged,
    p.total_refunded
FROM clean_orders o
LEFT JOIN (
    -- Match on restaurant_id and find the most recent effective_from date before or equal to order_ts
    -- We cast effective_from explicitly to TIMESTAMP to avoid comparison errors with order_ts
    SELECT
        r1.restaurant_id,
        r1.restaurant_name,
        r1.cuisine,
        r1.city,
        r1.effective_from::TIMESTAMP AS effective_from,
        LEAD(r1.effective_from::TIMESTAMP) OVER (PARTITION BY r1.restaurant_id ORDER BY r1.effective_from::TIMESTAMP) AS effective_to
    FROM clean_restaurants r1
) r ON o.restaurant_id = r.restaurant_id
   AND o.order_ts >= r.effective_from
   AND (o.order_ts < r.effective_to OR r.effective_to IS NULL)
LEFT JOIN clean_order_lines ol ON o.order_id = ol.order_id
LEFT JOIN clean_order_status_events ev ON o.order_id = ev.order_id
LEFT JOIN clean_payments p ON o.order_id = p.order_id
"""

orders_joined_df = q(join_query)
con.register('orders_joined', orders_joined_df)

print(f"Master joined table created with {len(orders_joined_df):,} rows.")
from IPython.display import display
display(orders_joined_df.head(5))

Master joined table created with 14,494 rows.


,order_id,customer_id,restaurant_id,order_ts,delivery_fee,restaurant_name,cuisine,restaurant_city,total_items_quantity,total_items_cost,...,accepted,picked_up,delivered,cancelled,final_status,duration_minutes,flag_negative_duration,flag_over_3_hours,total_charged,total_refunded
0,O000009,C02423,R046,2026-03-01 11:52:27+00:00,1.49,Happy Mexican 46,Mexican,Manchester,5,18.5,...,2026-03-01 11:56:42+00:00,NaT,2026-03-01 12:36:18+00:00,NaT,delivered,43.850000,False,False,19.99,0.0
1,O000013,C02671,R046,2026-03-01 12:15:09+00:00,0.00,Happy Mexican 46,Mexican,Manchester,2,21.0,...,2026-03-01 12:16:34+00:00,NaT,2026-03-01 13:04:08+00:00,NaT,delivered,48.983333,False,False,21.00,0.0
2,O000048,C03330,R046,2026-03-01 17:45:21+00:00,2.49,Happy Mexican 46,Mexican,Manchester,3,25.5,...,2026-03-01 17:47:43+00:00,NaT,2026-03-01 18:16:36+00:00,NaT,delivered,31.250000,False,False,27.99,0.0
3,O000049,C02047,R046,2026-03-01 17:57:35+00:00,2.99,Happy Mexican 46,Mexican,Manchester,6,50.5,...,2026-03-01 18:00:59+00:00,NaT,2026-03-01 18:29:20+00:00,NaT,delivered,31.750000,False,False,53.49,0.0
4,O000126,C00365,R046,2026-03-01 23:25:01+00:00,1.49,Happy Mexican 46,Mexican,Manchester,9,73.5,...,2026-03-01 23:29:03+00:00,NaT,2026-03-02 00:10:48+00:00,NaT,delivered,45.783333,False,False,74.99,0.0


In [16]:
# Build the final Gold Operations Report with Quality Flags
gold_weekly_report_query = """
SELECT
    DATE_TRUNC('week', order_ts) AS week,
    cuisine,
    restaurant_city AS city,
    COUNT(order_id) AS total_orders,
    SUM(total_charged - total_refunded) AS total_net_revenue,
    AVG(CASE WHEN final_status = 'delivered' THEN duration_minutes ELSE NULL END) AS avg_delivery_time_minutes,
    COUNT(CASE WHEN flag_negative_duration = TRUE OR flag_over_3_hours = TRUE THEN 1 END) AS suspicious_delivery_orders_count,
    COUNT(CASE WHEN total_refunded > total_charged THEN 1 END) AS suspicious_refund_orders_count
FROM orders_joined
WHERE final_status IN ('delivered', 'cancelled')
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;
"""

gold_report_df = q(gold_weekly_report_query)
con.register('gold_weekly_operations_report', gold_report_df)

print("--- Weekly Operations Report (Gold Layer) ---")
display(gold_report_df.head(10))

--- Weekly Operations Report (Gold Layer) ---


,week,cuisine,city,total_orders,total_net_revenue,avg_delivery_time_minutes,suspicious_delivery_orders_count,suspicious_refund_orders_count
0,2026-02-23 00:00:00+00:00,Burgers,Birmingham,4,125.99,38.812500,0,0
1,2026-02-23 00:00:00+00:00,Burgers,Bristol,4,144.15,40.200000,0,0
2,2026-02-23 00:00:00+00:00,Burgers,Leeds,1,99.98,NaN,0,0
3,2026-02-23 00:00:00+00:00,Burgers,London,11,396.42,39.683333,0,0
4,2026-02-23 00:00:00+00:00,Chinese,Birmingham,7,150.44,34.008333,0,0
5,2026-02-23 00:00:00+00:00,Chinese,London,8,301.91,35.073810,0,0
6,2026-02-23 00:00:00+00:00,Chinese,Manchester,1,24.99,33.666667,0,0
7,2026-02-23 00:00:00+00:00,Indian,Birmingham,1,43.99,42.633333,0,0
8,2026-02-23 00:00:00+00:00,Indian,London,8,348.30,30.673810,0,0
9,2026-02-23 00:00:00+00:00,Italian,Birmingham,8,150.94,36.242857,0,0


In [17]:
# Build a tracking table for orders with an 'unknown' status (no outcome)
unknown_status_query = """
SELECT
    order_id,
    customer_id,
    restaurant_id,
    order_ts,
    restaurant_name,
    cuisine,
    restaurant_city AS city,
    placed,
    accepted,
    picked_up,
    delivered,
    cancelled
FROM orders_joined
WHERE final_status = 'no outcome'
ORDER BY order_ts;
"""

unknown_orders_df = q(unknown_status_query)
con.register('unknown_status_orders', unknown_orders_df)

print(f"--- Unknown Status Orders Tracking Table ({len(unknown_orders_df):,} rows) ---")
display(unknown_orders_df.head(10))

--- Unknown Status Orders Tracking Table (215 rows) ---


,order_id,customer_id,restaurant_id,order_ts,restaurant_name,cuisine,city,placed,accepted,picked_up,delivered,cancelled
0,O000041,C01431,R021,2026-03-01 15:04:57+00:00,Grand Thai 21,Thai,Manchester,2026-03-01 15:04:57+00:00,2026-03-01 15:09:42+00:00,NaT,NaT,NaT
1,O000129,C02598,R053,2026-03-02 00:13:50+00:00,Royal Thai 53,Thai,Manchester,2026-03-02 00:13:50+00:00,2026-03-02 00:15:54+00:00,NaT,NaT,NaT
2,O000143,C01528,R019,2026-03-02 12:55:56+00:00,Spice Chinese 19,Chinese,London,2026-03-02 12:55:56+00:00,2026-03-02 12:59:11+00:00,NaT,NaT,NaT
3,O000189,C02227,R020,2026-03-02 19:11:43+00:00,Grand Burgers 20,Burgers,London,2026-03-02 19:11:43+00:00,2026-03-02 19:17:11+00:00,NaT,NaT,NaT
4,O000199,C01397,R051,2026-03-02 19:27:32+00:00,Spice Chinese 51,Chinese,Birmingham,2026-03-02 19:27:32+00:00,2026-03-02 19:32:36+00:00,NaT,NaT,NaT
5,O000207,C01791,R052,2026-03-02 19:41:44+00:00,Old Burgers 52,Burgers,London,2026-03-02 19:41:44+00:00,2026-03-02 19:45:59+00:00,NaT,NaT,NaT
6,O000256,C03104,R039,2026-03-03 12:40:35+00:00,Corner Japanese 39,Japanese,Bristol,2026-03-03 12:40:35+00:00,2026-03-03 12:42:13+00:00,NaT,NaT,NaT
7,O000302,C01273,R020,2026-03-03 19:04:03+00:00,Grand Burgers 20,Burgers,London,2026-03-03 19:04:03+00:00,2026-03-03 19:08:30+00:00,NaT,NaT,NaT
8,O000314,C02113,R021,2026-03-03 19:28:32+00:00,Grand Thai 21,Thai,Manchester,2026-03-03 19:28:32+00:00,2026-03-03 19:30:58+00:00,NaT,NaT,NaT
9,O000420,C02784,R043,2026-03-04 19:56:40+00:00,Old Chinese 43,Chinese,London,2026-03-04 19:56:40+00:00,2026-03-04 19:58:15+00:00,NaT,NaT,NaT


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.